In [2]:
import argparse
import sys
from pathlib import Path

import cv2

#import nrrd
import gc

import numpy as np

import os
import SimpleITK as sitk
from scipy.interpolate import RegularGridInterpolator

import pydicom as dicom

import matplotlib.pyplot as plt
from scipy.ndimage import zoom
from scipy import ndimage

import glob

# !pip install nibabel
import nibabel as nib


In [3]:
def multires_registration(fixed_image, moving_image,initial_transform):
        
    registration_method = sitk.ImageRegistrationMethod()
    registration_method.SetInterpolator(sitk.sitkBSpline)
    registration_method.SetMetricAsMattesMutualInformation(numberOfHistogramBins=20)
    registration_method.SetMetricSamplingStrategy(registration_method.RANDOM)
    registration_method.SetMetricSamplingPercentage(0.05)
    registration_method.SetOptimizerAsGradientDescent(learningRate=0.1, numberOfIterations=200, convergenceMinimumValue=1e-8, convergenceWindowSize=2)
    registration_method.SetOptimizerScalesFromPhysicalShift() 
    registration_method.SetShrinkFactorsPerLevel(shrinkFactors = [8,4,2,1]) 
    registration_method.SetSmoothingSigmasPerLevel(smoothingSigmas = [3,2,1,0]) 
    registration_method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    optimized_transform = sitk.AffineTransform(3)
    registration_method.SetMovingInitialTransform(initial_transform)  
    registration_method.SetInitialTransform(optimized_transform, inPlace=False)
    optimized_transform = registration_method.Execute(fixed_image, moving_image)
        

    return (optimized_transform)

def smooth_and_resample(image, shrink_factor, smoothing_sigma):
    """
    Args:
        image: The image we want to resample.
        shrink_factor: A number greater than one, such that the new image's size is original_size/shrink_factor.
        smoothing_sigma: Sigma for Gaussian smoothing, this is in physical (image spacing) units, not pixels.
    Return:
        Image which is a result of smoothing the input and then resampling it using the given sigma and shrink factor.
    """
    if smoothing_sigma>0:
        smoothed_image = sitk.SmoothingRecursiveGaussian(image, smoothing_sigma)
    else:
        smoothed_image = image
    
    original_spacing = image.GetSpacing()
    original_size = image.GetSize()
    new_size = [int(sz/float(shrink_factor) + 0.5) for sz in original_size]
    new_spacing = [((original_sz-1)*original_spc)/(new_sz-1) 
                   for original_sz, original_spc, new_sz in zip(original_size, original_spacing, new_size)]
    return sitk.Resample(smoothed_image, new_size, sitk.Transform(), 
                         sitk.sitkLinear, image.GetOrigin(),
                         new_spacing, image.GetDirection(), 0.0, 
                         image.GetPixelID())


    
def multiscale_demons(registration_algorithm,
                      fixed_image, moving_image, initial_transform = None, 
                      shrink_factors=None, smoothing_sigmas=None):
    """
    Run the given registration algorithm in a multiscale fashion. The original scale should not be given as input as the
    original images are implicitly incorporated as the base of the pyramid.
    Args:
        registration_algorithm: Any registration algorithm that has an Execute(fixed_image, moving_image, displacement_field_image)
                                method.
        fixed_image: Resulting transformation maps points from this image's spatial domain to the moving image spatial domain.
        moving_image: Resulting transformation maps points from the fixed_image's spatial domain to this image's spatial domain.
        initial_transform: Any SimpleITK transform, used to initialize the displacement field.
        shrink_factors: Shrink factors relative to the original image's size.
        smoothing_sigmas: Amount of smoothing which is done prior to resmapling the image using the given shrink factor. These
                          are in physical (image spacing) units.
    Returns: 
        SimpleITK.DisplacementFieldTransform
    """
    # Create image pyramid.
    fixed_images = [fixed_image]
    moving_images = [moving_image]
    if shrink_factors:
        for shrink_factor, smoothing_sigma in reversed(list(zip(shrink_factors, smoothing_sigmas))):
            fixed_images.append(smooth_and_resample(fixed_images[0], shrink_factor, smoothing_sigma))
            moving_images.append(smooth_and_resample(moving_images[0], shrink_factor, smoothing_sigma))
    
    # Create initial displacement field at lowest resolution. 
    # Currently, the pixel type is required to be sitkVectorFloat64 because of a constraint imposed by the Demons filters.
    if initial_transform:
        initial_displacement_field = sitk.TransformToDisplacementField(initial_transform, 
                                                                       sitk.sitkVectorFloat64,
                                                                       fixed_images[-1].GetSize(),
                                                                       fixed_images[-1].GetOrigin(),
                                                                       fixed_images[-1].GetSpacing(),
                                                                       fixed_images[-1].GetDirection())
    else:
        initial_displacement_field = sitk.Image(fixed_images[-1].GetWidth(), 
                                                fixed_images[-1].GetHeight(),
                                                fixed_images[-1].GetDepth(),
                                                sitk.sitkVectorFloat64)
        initial_displacement_field.CopyInformation(fixed_images[-1])
 
    # Run the registration.            
    initial_displacement_field = registration_algorithm.Execute(fixed_images[-1], 
                                                                moving_images[-1], 
                                                                initial_displacement_field)
    # Start at the top of the pyramid and work our way down.    
    for f_image, m_image in reversed(list(zip(fixed_images[0:-1], moving_images[0:-1]))):
            initial_displacement_field = sitk.Resample (initial_displacement_field, f_image)
            initial_displacement_field = registration_algorithm.Execute(f_image, m_image, initial_displacement_field)
    return sitk.DisplacementFieldTransform(initial_displacement_field)

def interp3(x,y,z,img_org, xr,yr,zr):
    """resampling"""
    
    xi,yi,zi = np.meshgrid(xr, yr, zr, indexing='ij')
    min_val = np.min(img_org)

    interp = RegularGridInterpolator((x,y,z),img_org,bounds_error=False,fill_value=min_val)
    img = interp( (xi,yi,zi)).astype(np.float32)
    
    return img


def dicom2vol(path):
    """dicom folder to 3-D image"""
    image_shape = []
    path = str(path)
    dicom_files = glob.glob(os.path.join(path,'**/*.dcm'), recursive = True)
    if len(dicom_files) ==0:
        dicom_files = glob.glob(os.path.join(path,'**/*.IMA'), recursive = True)
    slices = [dicom.read_file(s, force=True) for s in dicom_files]
    slices = sorted(slices,key=lambda x:x.ImagePositionPatient[2])
    pixel_spacing = slices[0].PixelSpacing
    slice_thickness = np.abs(slices[0].SliceLocation-slices[-1].SliceLocation)/np.float32(len(slices)-1.0)
    
    tmp_shape = list(slices[0].pixel_array.shape)
    image_shape.append(len(slices))
    image_shape.append(tmp_shape[0])
    image_shape.append(tmp_shape[1])
    
    image_shape = np.array(image_shape).astype(np.int32)
    
    volume3d = np.zeros(image_shape)
    for i,s in enumerate(slices):
        array2d = s.pixel_array
        volume3d[i] = array2d*s.RescaleSlope + s.RescaleIntercept

    ## x,y,z -> z,y,x
    volume3d = np.moveaxis(volume3d,0,-1)
    volume3d = np.moveaxis(volume3d,0,1)
    
    image_res = [pixel_spacing[1], pixel_spacing[0], slice_thickness]

        
    return volume3d, image_res

def dicom2nifty(path_dicom):
    
    volume3d, image_res = dicom2vol(path_dicom)
    
    affine_matrix = np.eye(4)
    affine_matrix[0,0] = image_res[0]
    affine_matrix[1,1] = image_res[1]
    affine_matrix[2,2] = image_res[2]
    
    img = nib.Nifti1Image(volume3d[::-1,::-1,:], affine_matrix)
    
    nib.save(img,os.path.join(path_dicom + '_org.nii.gz'))
    
    return np.moveaxis(volume3d,0,1), image_res


   

In [ ]:
def registration(path_A, path_P, path_D):
    """Resampling and registation"""

    # Resampling parameters: 
    # voxel size: 0.8 mm x 0.8 mm x 2.5 mm
    # 3-D image: 384 x 384 x 96    
    ref_dx, ref_dy, ref_dz = 0.8, 0.8, 2.5
    ref_nx, ref_ny, ref_nz = 384, 384, 96

    xr = (np.linspace(0, ref_nx-1, ref_nx) - (ref_nx-1)/2.0)*ref_dx
    yr = (np.linspace(0, ref_ny-1, ref_ny) - (ref_ny-1)/2.0)*ref_dy
    zr = (np.linspace(0, ref_nz-1, ref_nz) - (ref_nz-1)/2.0)*ref_dz

    # registation paramter
    demons_filter =  sitk.FastSymmetricForcesDemonsRegistrationFilter()
    demons_filter.SetNumberOfIterations(50)
    demons_filter.SetSmoothDisplacementField(True)
    demons_filter.SetStandardDeviations(2.0)
    
    A, A_res = dicom2nifty(path_A) 
    P, P_res = dicom2nifty(path_P) 
    D, D_res = dicom2nifty(path_D) 

    A_shape = A.shape
    P_shape = P.shape
    D_shape = D.shape
    
    nz = np.minimum(A_shape[2],D_shape[2])
    
    A = A[:,:,-nz:]
    P = P[:,:,-nz:]
    D = D[:,:,-nz:]

    x = (np.linspace(0, P_shape[0]-1, P_shape[0]) - (P_shape[0]-1)/2.0)*P_res[0]
    y = (np.linspace(0, P_shape[1]-1, P_shape[1]) - (P_shape[1]-1)/2.0)*P_res[1]
    z = (np.linspace(0, nz-1, nz) - (nz-1)/2.0)*P_res[2]
    
    img_P = interp3(x, y, z, P, xr, yr, zr)
    img_P = np.moveaxis(img_P, 2, 0)
    
    fixed_image = sitk.GetImageFromArray((img_P).astype(np.float32))
    fixed_image.SetSpacing((ref_dx,ref_dy,ref_dz))
    
    sitk.WriteImage(fixed_image,os.path.join(os.path.split(str(path_P))[0],'P.nii.gz'))

    x = (np.linspace(0, A_shape[0]-1, A_shape[0]) - (A_shape[0]-1)/2.0)*A_res[0]
    y = (np.linspace(0, A_shape[1]-1, A_shape[1]) - (A_shape[1]-1)/2.0)*A_res[1]
    z = (np.linspace(0, nz-1, nz) - (nz-1)/2.0)*A_res[2]

    img_A = interp3(x, y, z, A, xr, yr, zr)
    img_A = np.moveaxis(img_A, 2, 0)
    
    moving_image = sitk.GetImageFromArray((img_A).astype(np.float32))
    moving_image.SetSpacing((ref_dx,ref_dy,ref_dz))

    initial_transform = sitk.CenteredTransformInitializer(  fixed_image, 
                                                            moving_image, 
                                                            sitk.AffineTransform(3),
                                                            sitk.CenteredTransformInitializerFilter.GEOMETRY)

    optimized_transform = multires_registration(fixed_image, moving_image,initial_transform)
    affine_transform = sitk.Transform(optimized_transform)

    # Run the registration.
    tx = multiscale_demons(registration_algorithm=demons_filter, 
                           fixed_image = fixed_image, 
                           moving_image = moving_image,
                           initial_transform = affine_transform,
                           shrink_factors = [2,1],
                           smoothing_sigmas = [1,0])

    moving_reg = sitk.Resample(moving_image, fixed_image, tx, sitk.sitkLinear, 0.0, moving_image.GetPixelID())
    moving_reg.SetSpacing((ref_dx, ref_dy, ref_dz))

    sitk.WriteImage(moving_reg,os.path.join(os.path.split(str(path_A))[0],'A.nii.gz'))
    
    # ============================= #
    
    x = (np.linspace(0, D_shape[0]-1, D_shape[0]) - (D_shape[0]-1)/2.0)*D_res[0]
    y = (np.linspace(0, D_shape[1]-1, D_shape[1]) - (D_shape[1]-1)/2.0)*D_res[1]
    z = (np.linspace(0, nz-1, nz) - (nz-1)/2.0)*D_res[2]

    img_D = interp3(x, y, z, D, xr, yr, zr)
    img_D = np.moveaxis(img_D, 2, 0)
    
    moving_image = sitk.GetImageFromArray((img_D).astype(np.float32))
    moving_image.SetSpacing((ref_dx,ref_dy,ref_dz))

    initial_transform = sitk.CenteredTransformInitializer(  fixed_image, 
                                                            moving_image, 
                                                            sitk.AffineTransform(3),
                                                            sitk.CenteredTransformInitializerFilter.GEOMETRY)

    optimized_transform = multires_registration(fixed_image, moving_image,initial_transform)
    affine_transform = sitk.Transform(optimized_transform)

    # Run the registration.
    tx = multiscale_demons(registration_algorithm=demons_filter, 
                           fixed_image = fixed_image, 
                           moving_image = moving_image,
                           initial_transform = affine_transform,
                           shrink_factors = [2,1],
                           smoothing_sigmas = [1,0])

    moving_reg = sitk.Resample(moving_image, fixed_image, tx, sitk.sitkLinear, 0.0, moving_image.GetPixelID())
    moving_reg.SetSpacing((ref_dx, ref_dy, ref_dz))

    sitk.WriteImage(moving_reg,os.path.join(os.path.split(str(path_D))[0],'D.nii.gz'))

## Step 1: Resampling and Registration

In [6]:
path_A = '20180320/Arterial'
path_P = '20180320/Portal'
path_D = '20180320/Delayed'

registration(path_A, path_P, path_D)

AttributeError: module 'pydicom' has no attribute 'read_file'

## Step 2: Liver segmentation using total segmentator

In [26]:
from totalsegmentator.python_api import totalsegmentator

In [27]:
path_img = '20180320/P.nii.gz'
path_seg = '20180320/P_seg.nii.gz'
totalsegmentator(path_img, path_seg, ml=True, fast=True)


If you use this tool please cite: https://doi.org/10.48550/arXiv.2208.05868

No GPU detected. Running on CPU. This can be very slow. The '--fast' option can help to some extend.
Using 'fast' option: resampling to lower resolution (3mm)
Resampling...
  Resampled in 0.88s
Predicting...


C:\Users\kkim\anaconda3\envs\tensorflow_2_10\lib\site-packages\torch\cuda\amp\grad_scaler.py:120: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn("torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.")
C:\Users\kkim\anaconda3\envs\tensorflow_2_10\lib\site-packages\torch\amp\autocast_mode.py:204: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn('User provided device_type of \'cuda\', but CUDA is not available. Disabling')


  Predicted in 36.01s
Resampling...
Saving segmentations...
  Saved in 0.32s


## Step 3: Liver cancer segmentation using 3 phase CT images

In [32]:
from UNet3D import unet3D

import numpy as np

import tensorflow as tf

from tensorflow.keras import backend as K

def normalize_inten(img):
    
    center = 45
    window = 300
    
    max_val = center+window/2
    min_val = center-window/2
    img_norm = np.clip( (img-min_val)/(max_val-min_val),0,1)*250
    
    return img_norm.astype(np.uint8)

def dice_coef_oneclass(y_true, y_pred, smooth):
    # Flatten predictions, preserving the class dimension
    y_pred = K.batch_flatten(y_pred)
    y_true = K.batch_flatten(y_true)
    
    class_intersection = K.sum(y_true * y_pred)
    class_loss = (2. * class_intersection + smooth) / (K.sum(y_true) + K.sum(y_pred) + smooth)
        
    return class_loss


def dice_coef_oneclass_bias(y_true, y_pred, smooth, bias):
    y_pred = K.batch_flatten(y_pred)
    y_true = K.batch_flatten(y_true)
    
    class_intersection = K.sum(y_true * y_pred)**bias
    class_loss = (2.0 * class_intersection + smooth) / (K.sum(y_true) + K.sum(y_pred) + smooth)
        
    return class_loss


def soft_dice_loss(y_true, y_pred):    
    return 1 - dice_coef_oneclass(y_true, y_pred, 1.0)


def soft_dice_rmse_loss(y_true, y_pred):
    alpha = 1
    RMSE_loss = K.sqrt(K.mean(K.square(y_true - y_pred)))
    return RMSE_loss+ alpha*(1 - dice_coef_oneclass_bias(y_true, y_pred, 1.0, 1.0))


def hard_dice_coef(y_true, y_pred):
    y_pred = K.cast(K.greater(y_pred, 0.5), 'float32')
    return dice_coef_oneclass(y_true, y_pred, 1e-4)


def IOU_loss(y_true, y_pred):
    y_pred = K.batch_flatten(y_pred)
    y_true = K.batch_flatten(y_true)
    
    class_intersection = K.sum(y_true * y_pred)
    class_loss = (class_intersection + 1) / (K.sum(y_true) + K.sum(y_pred) - class_intersection + 1)
    return 1 - class_loss 


def get_augmentation(patch_size):
    """Get augmentation."""
    return Compose([
        Rotate((-5, 5), (-5, 5), (0, 0), p=0.5),
#         RandomCropFromBorders(crop_value=0.1, p=0.5),
        ElasticTransform((0, 0.05), interpolation=2, p=0.1),
        RandomScale(scale_limit=[0.95, 1.05], interpolation=1, always_apply=True, p=1.0),
#         Flip(0, p=0.5),
#         Flip(1, p=0.5),
#         Flip(2, p=0.5),
#         RandomRotate90((1, 2), p=0.5),
#         GaussianNoise(var_limit=(0, 5), p=0.2),
#         RandomGamma(gamma_limit=(0.5, 1.5), p=0.2),
    ], p=1.0)


def load_dataset(path_src):
    """Load data."""
    imgs = np.load(str(path_src) + '_img.npy',mmap_mode='r')
    labels = np.load(str(path_src) + '_label.npy',mmap_mode='r')

    labels = np.expand_dims(labels, axis=-1).astype(np.float32)

    return imgs, labels

def data_aug(img, label):
    shift_range_x = 32
    shift_range_y = 32
    shift_range_z = 16
    len_x = 64
    len_y = 64
    len_z = 16
    xs = np.random.randint(0,shift_range_x,img.shape[0])
    ys = np.random.randint(0,shift_range_y,img.shape[0])
    zs = xs = np.random.randint(0,shift_range_z,img.shape[0])
    img_aug = []
    label_aug = []
    for i in range(img.shape[0]):
        img_aug.append(img[i,xs[i]:xs[i]+len_x,ys[i]:ys[i]+len_y,zs[i]:zs[i]+len_z,:])
        label_aug.append(label[i,xs[i]:xs[i]+len_x,ys[i]:ys[i]+len_y,zs[i]:zs[i]+len_z,:])
     
    img_aug = np.array(img_aug)
    label_aug = np.array(label_aug)
    
    return img_aug, label_aug


def img2patch(img, px, py, pz):
    """3-D image to patches with a size of (px,py,pz)"""
    
    nx, ny, nz, nc = img.shape
    
    shift_range_x = np.int32(px/2)
    shift_range_y = np.int32(py/2)
    shift_range_z = np.int32(pz/2)
    
    img_aug = []
    
    rx = range(0,nx-px+1,shift_range_x)
    ry = range(0,ny-py+1,shift_range_y)
    rz = range(0,nz-pz+1,shift_range_z)

    for ix in rx:
        for iy in ry:
            for iz in rz:
                
                img_aug.append(np.squeeze(img[ix:ix+px,iy:iy+py,iz:iz+pz,:]))
                
    return np.array(img_aug)

def patch2img(patch, nx, ny, nz):
    """patches to 3-D image with a size of (nx,ny,nz)"""
    
    nn, px, py, pz = patch.shape
    
    shift_range_x = np.int32(px/2)
    shift_range_y = np.int32(py/2)
    shift_range_z = np.int32(pz/2)
    
    img = np.zeros( (nx, ny, nz), dtype=float)
    norm = np.zeros( (nx, ny, nz), dtype=float)
    
    patch_ones = np.ones( (px, py, pz), dtype=float)
    
    rx = range(0,nx-px+1,shift_range_x)
    ry = range(0,ny-py+1,shift_range_y)
    rz = range(0,nz-pz+1,shift_range_z)

    ip = 0
    for ix in rx:
        for iy in ry:
            for iz in rz:
                
                img[ix:ix+px,iy:iy+py,iz:iz+pz] += patch[ip]
                norm[ix:ix+px,iy:iy+py,iz:iz+pz] += patch_ones
                
                ip += 1
                
    img = img/norm
    
    return np.array(img)


def detection(path_A, path_P, path_D, path_model_weight):
    """Test detection model."""
    print('Running liver lesion detection')
    print('File name of model weight: %s' % path_model_weight)

    tf.keras.backend.clear_session()
    
    # patch size for detection
    px = 64
    py = 64
    pz = 16
    
    img_A = sitk.ReadImage(os.path.join(str(path_A)))
    img_P = sitk.ReadImage(os.path.join(str(path_P)))
    img_D = sitk.ReadImage(os.path.join(str(path_D)))

    nx, ny, nz = img_A.GetSize()
    dx, dy, dz = img_A.GetSpacing()

    A = sitk.GetArrayFromImage(img_A)
    P = sitk.GetArrayFromImage(img_P) 
    D = sitk.GetArrayFromImage(img_D)
    
    A = np.array(A)
    P = np.array(P)
    D = np.array(D)
    
#     print(A.shape)
    
    A = np.moveaxis(A,0,-1)
    P = np.moveaxis(P,0,-1)
    D = np.moveaxis(D,0,-1)
    
#     print(A.shape)
    
    img = []
    img.append(A)
    img.append(P)
    img.append(D)
    
    
    
    img = np.array(img)
    img = np.moveaxis(img,0,-1)
    
    model_params = {'num_convs': 2,
         'compression_channels': [12, 24, 48, 96, 256],
         'decompression_channels': [96, 48, 24, 12],
         'compression_dropout': None,
         'decompression_dropout': [None, None, None, None],
         'output_activation': 'sigmoid',
         'conv_kwargs': {'kernel_regularizer': tf.keras.regularizers.l2(1e-3)},
    }    

    unet3D_model = unet3D.get_unet_3D(num_classes=1, input_shape=(px,py,pz,3), **model_params)
    unet3D_model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=1.0e-5, amsgrad = True),
                         loss = IOU_loss, metrics=[hard_dice_coef])
    
    unet3D_model.load_weights(str(path_model_weight))
    
    patch_img = img2patch(normalize_inten(img), px, py, pz)
#     print(img.shape)
#     print(patch_img.shape)
    pred_patch_img = unet3D_model.predict(patch_img.reshape(patch_img.shape[0],px,py,pz,3).astype(np.uint8),batch_size=1)

    print(pred_patch_img.shape)
    
    pred_img = patch2img(np.squeeze(pred_patch_img), nx, ny, nz)
    
    pred_img *= 250.0

    det_image = sitk.GetImageFromArray(np.moveaxis(np.squeeze(pred_img),-1,0).astype(np.float32))
    det_image.SetSpacing((dx,dy,dz))

    sitk.WriteImage(det_image,os.path.join(os.path.split(str(path_P))[0],'target.nii.gz'))

In [37]:
path_A = '20180320/A.nii.gz'
path_P = '20180320/P.nii.gz'
path_D = '20180320/D.nii.gz'

path_model_weight = 'model_unet3D_64.h5'

detection(path_A, path_P, path_D, path_model_weight)

Running liver lesion detection
File name of model weight: model_unet3D_64.h5
1331/1331 [==============================] - 23s 17ms/step
(1331, 64, 64, 16, 1)


In [42]:
organ_seg = sitk.ReadImage('20180320/P_seg.nii.gz')
tumor_seg = sitk.ReadImage('20180320/target.nii.gz')

organ_img = sitk.GetArrayFromImage(organ_seg)
tumor_img = sitk.GetArrayFromImage(tumor_seg)

# # organ segmentation: liver == 5 -> we only care liver area.
tumor_img[np.where(organ_img != 5)] = 0
tumor_img[np.where(tumor_img <= 60)] = 0
tumor_img = np.clip(tumor_img/120,0,1)

tumor = sitk.GetImageFromArray(tumor_img)
tumor.SetSpacing(tumor_seg.GetSpacing())

sitk.WriteImage(tumor,os.path.join(os.path.split(str(path_P))[0],'tumor.nii.gz'))
